# Providers 03 - vLLM ModelServer (CLI)

Notebook CLI paralelo a `tutorials/providers/03_vllm.ipynb`.

**Objetivo:** Inspeccionar el comando exacto de vLLM + Unsloth sin iniciar procesos.

Este notebook no llama factories de Agentic Systems directamente: ejecuta el
entrypoint CLI real, conserva la salida Rich y valida después el JSON del mismo
contrato.


## Cómo se ejecuta

La forma portable es `python -m agentic_systems.cli ...`. Después de instalar
el wheel, el entrypoint equivalente es `agentic-systems ...`.


In [1]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path


def _repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError("No se encontró la raíz del repositorio.")


ROOT = _repo_root()
CLI = [sys.executable, "-m", "agentic_systems.cli"]


def run_cli(*args: str, expected: int = 0) -> str:
    env = os.environ.copy()
    source_path = str(ROOT / "src")
    env["PYTHONPATH"] = (
        source_path
        if not env.get("PYTHONPATH")
        else source_path + os.pathsep + env["PYTHONPATH"]
    )
    command = [*CLI, *args]
    completed = subprocess.run(
        command,
        cwd=ROOT,
        env=env,
        text=True,
        encoding="utf-8",
        errors="replace",
        capture_output=True,
        check=False,
    )
    print("$ " + " ".join(command))
    if completed.stdout:
        print(completed.stdout, end="")
    if completed.stderr:
        print(completed.stderr, end="")
    assert completed.returncode == expected, completed.stderr
    return completed.stdout


def run_cli_json(*args: str) -> dict:
    return json.loads(run_cli(*args, "--json"))


def assert_rich(output: str, title: str) -> None:
    assert title in output
    ascii_box = "+" in output and "|" in output
    unicode_box = "─" in output and "│" in output
    assert ascii_box or unicode_box


## 1) Salida humana Rich

La celda conserva stdout y comprueba título y bordes. Esto detecta tablas o
paneles truncados, además del exit code.


In [2]:
rich_output = run_cli(*['model-server', 'inspect', '--model', 'unsloth/Qwen3-0.6B', '--profile', 'fast', '--reasoning-parser', 'qwen3'])
assert_rich(rich_output, 'Model Server')


$ C:\Python314\python.exe -m agentic_systems.cli model-server inspect --model unsloth/Qwen3-0.6B --profile fast --reasoning-parser qwen3
+------------------------------- Model Server --------------------------------+
| {                                                                           |
|   "backend": "vllm",                                                        |
|   "command": [                                                              |
|     "vllm",                                                                 |
|     "serve",                                                                |
|     "unsloth/Qwen3-0.6B",                                                   |
|     "--host",                                                               |
|     "127.0.0.1",                                                            |
|     "--port",                                                               |
|     "8000",                                                  

## 2) Contrato de máquina

La misma ruta se ejecuta con `--json` para afirmar campos y cardinalidad sin
parsear la presentación Rich.


In [3]:
payload = run_cli_json(*['model-server', 'inspect', '--model', 'unsloth/Qwen3-0.6B', '--profile', 'fast', '--reasoning-parser', 'qwen3'])
assert payload["backend"] == "vllm" and payload["spec"]["profile"] == "fast"
payload


$ C:\Python314\python.exe -m agentic_systems.cli model-server inspect --model unsloth/Qwen3-0.6B --profile fast --reasoning-parser qwen3 --json
{
  "backend": "vllm",
  "command": [
    "vllm",
    "serve",
    "unsloth/Qwen3-0.6B",
    "--host",
    "127.0.0.1",
    "--port",
    "8000",
    "--served-model-name",
    "unsloth/Qwen3-0.6B",
    "--gpu-memory-utilization",
    "0.4",
    "--max-model-len",
    "2048",
    "--max-num-seqs",
    "4",
    "--enable-auto-tool-choice",
    "--tool-call-parser",
    "hermes",
    "--reasoning-parser",
    "qwen3",
    "--generation-config",
    "vllm"
  ],
  "endpoint": {
    "api_key_configured": true,
    "backend": "vllm",
    "base_url": "http://127.0.0.1:8000/v1",
    "model_id": "unsloth/Qwen3-0.6B",
    "owned": false,
    "pid": null,
    "schema_version": "agentic_systems.serving.v1"
  },
  "installed": false,
  "spec": {
    "artifact": {
      "adapter_path": null,
      "base_model_id": null,
      "metadata": {},
      "model_id"

{'backend': 'vllm',
 'command': ['vllm',
  'serve',
  'unsloth/Qwen3-0.6B',
  '--host',
  '127.0.0.1',
  '--port',
  '8000',
  '--served-model-name',
  'unsloth/Qwen3-0.6B',
  '--gpu-memory-utilization',
  '0.4',
  '--max-model-len',
  '2048',
  '--max-num-seqs',
  '4',
  '--enable-auto-tool-choice',
  '--tool-call-parser',
  'hermes',
  '--reasoning-parser',
  'qwen3',
  '--generation-config',
  'vllm'],
 'endpoint': {'api_key_configured': True,
  'backend': 'vllm',
  'base_url': 'http://127.0.0.1:8000/v1',
  'model_id': 'unsloth/Qwen3-0.6B',
  'owned': False,
  'pid': None,
  'schema_version': 'agentic_systems.serving.v1'},
 'installed': False,
 'spec': {'artifact': {'adapter_path': None,
   'base_model_id': None,
   'metadata': {},
   'model_id': 'unsloth/Qwen3-0.6B',
   'quantization': None,
   'revision': None,
   'schema_version': 'agentic_systems.serving.v1',
   'tokenizer_id': None},
  'backend': 'vllm',
  'binary': 'vllm',
  'enable_auto_tool_choice': True,
  'extra_args': [],

## Resultado e interpretación

Rich responde a lectura humana; JSON responde a automatización. Ambos nacen del
mismo comando y del mismo escenario público. Un estado `not-run` conserva el
motivo, pero no cuenta como evidencia live.
